# 💳 Credit Card Default Prediction with XGBoost
### Cost-Sensitive Classification — UCI Default of Credit Card Clients Dataset

## 📌 Overview
A financial institution wants to predict **which credit card customers are
likely to default** on their payment next month, using their demographic
info and 6 months of billing/payment history.

Unlike a standard "maximize accuracy" classification task, this is a
**cost-sensitive problem**: missing an actual default (false negative) costs
the bank far more than flagging a good customer as risky (false positive).
This project uses **XGBoost** — tuned specifically for imbalanced data and
business-driven threshold selection, not just raw accuracy.

## 🎯 What Makes This Project Challenging
- **Imbalanced target** (~22% default rate) — accuracy alone is misleading
- **Cost-sensitive decision making** — tuning the classification threshold
  based on real business cost, not the

## 📦 Step 1: Setup & Configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

THEME = {
    "bg": "#0d0221",
    "primary": "#7c3aed",
    "light": "#c4b5fd",
    "accent": "#f472b6",
    "text": "#c4b5fd",
}

def apply_dark_theme(ax, title, xlabel="", ylabel=""):
    """Apply consistent dark purple theme to a matplotlib axis."""
    ax.set_facecolor(THEME["bg"])
    ax.set_title(title, fontsize=15, color="white", pad=15)
    ax.set_xlabel(xlabel, color=THEME["light"])
    ax.set_ylabel(ylabel, color=THEME["light"])
    ax.tick_params(colors=THEME["light"])
    ax.grid(True, alpha=0.2, color=THEME["primary"])

plt.rcParams["figure.facecolor"] = THEME["bg"]

## 📂 Step 2: Load Data

In [7]:
df = pd.read_csv("default of credit card clients.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 🔍 Step 3: Initial Exploration

Before cleaning or engineering anything, we check data types, missing
values, and the target distribution to understand what we're working with.

In [8]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum().sum())
print("\nTarget distribution:")
print(df["default payment next month"].value_counts(normalize=True))

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_AMT3        

## 🧹 Step 4: Check for Invalid Categorical Values

The dataset documentation defines specific valid categories for `EDUCATION`
and `MARRIAGE`, but real-world exports often contain undocumented codes.
We check for these before deciding how to handle them.

In [9]:
print("EDUCATION unique values:", sorted(df["EDUCATION"].unique()))
print("MARRIAGE unique values:", sorted(df["MARRIAGE"].unique()))
print("\nEDUCATION value counts:\n", df["EDUCATION"].value_counts().sort_index())
print("\nMARRIAGE value counts:\n", df["MARRIAGE"].value_counts().sort_index())

EDUCATION unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
MARRIAGE unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

EDUCATION value counts:
 EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

MARRIAGE value counts:
 MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64


## 🧹 Step 5: Clean Invalid Categorical Values

`EDUCATION` codes `0`, `5`, `6` and `MARRIAGE` code `0` are undocumented.
Since both features already have an explicit "Others" category (`EDUCATION=4`,
`MARRIAGE=3`), we merge the undocumented codes into "Others" rather than
dropping rows — this preserves data while keeping categories meaningful.

In [11]:
# EDUCATION: merge 0, 5, 6 into 4 (Others)
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})

# MARRIAGE: merge 0 into 3 (Others)
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})

print("EDUCATION after cleaning:\n", df["EDUCATION"].value_counts().sort_index())
print("\nMARRIAGE after cleaning:\n", df["MARRIAGE"].value_counts().sort_index())

EDUCATION after cleaning:
 EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64

MARRIAGE after cleaning:
 MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64


## 🗑️ Step 6: Drop Identifier Column

`ID` is just a row identifier with no predictive value — keeping it risks
the model accidentally learning spurious patterns from it.

In [12]:
df = df.drop("ID", axis=1)
print(f"Shape: {df.shape}")

Shape: (30000, 24)


## 🔍 Step 7: Explore Payment History Features

The dataset includes 6 months of repayment status (`PAY_0`, `PAY_2`...`PAY_6`),
billed amount (`BILL_AMT1`...`BILL_AMT6`), and paid amount
(`PAY_AMT1`...`PAY_AMT6`). Understanding their scale and distribution helps
us decide what engineered features might be useful later.

Per the dataset documentation, `PAY_*` values mean:
- `-1` = paid duly, `1` = 1 month delay, `2` = 2 months delay, ... `9` = 9+ months delay
- `0` is undocumented but commonly interpreted as "paid minimum / no delay recorded"

In [13]:
pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
payamt_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

print("PAY_* unique values across all months:")
print(sorted(df[pay_cols].values.flatten().tolist()))
print(set(df[pay_cols].values.flatten().tolist()))

print("\nBILL_AMT summary:\n", df[bill_cols].describe().T[["mean", "min", "max"]])
print("\nPAY_AMT summary:\n", df[payamt_cols].describe().T[["mean", "min", "max"]])

PAY_* unique values across all months:
[-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, 

## ⚙️ Step 8: Feature Engineering

Raw monthly columns are useful, but derived behavioral signals often carry
more predictive power for a tree-based model:

- **Average / trend of delay status** across the 6 months — is the customer
  consistently late, or was it a one-time issue?
- **Utilization ratio** — how much of their credit limit are they actually
  using? (`BILL_AMT` relative to `LIMIT_BAL`)
- **Payment ratio** — how much of their bill are they actually paying off
  each month? (`PAY_AMT` relative to `BILL_AMT`)
- **Total months delayed** — a simple count of how many of the 6 months had
  a delay flag (`PAY_* > 0`)

In [15]:
# Average delay status across the 6 months (higher = more consistently late)
df["avg_pay_status"] = df[pay_cols].mean(axis=1)

# How many of the 6 months had an actual delay (PAY_* > 0)
df["months_delayed"] = (df[pay_cols] > 0).sum(axis=1)

# Most recent delay status (PAY_0 = most recent month) is often the strongest signal
df["recent_delay"] = df["PAY_0"]

# Average credit utilization: how much of their limit they're carrying as debt
df["avg_utilization"] = df[bill_cols].mean(axis=1) / df["LIMIT_BAL"]

# Average payment ratio — total paid across 6 months relative to total billed
# (more stable than averaging 6 separate ratios, which can blow up when a
# single month's bill is near zero)
total_bill = df[bill_cols].sum(axis=1)
total_paid = df[payamt_cols].sum(axis=1)
df["avg_payment_ratio"] = (total_paid / total_bill.replace(0, np.nan)).fillna(0)
df["avg_payment_ratio"] = df["avg_payment_ratio"].clip(lower=-2, upper=2)

print(df[["avg_pay_status", "months_delayed", "recent_delay", "avg_utilization", "avg_payment_ratio"]].describe())

       avg_pay_status  months_delayed  recent_delay  avg_utilization  \
count    30000.000000    30000.000000  30000.000000     30000.000000   
mean        -0.182439        0.834200     -0.016700         0.373048   
std          0.982176        1.554303      1.123802         0.351890   
min         -2.000000        0.000000     -2.000000        -0.232590   
25%         -0.833333        0.000000     -1.000000         0.029997   
50%          0.000000        0.000000      0.000000         0.284834   
75%          0.000000        1.000000      0.000000         0.687929   
max          6.000000        6.000000      8.000000         5.364308   

       avg_payment_ratio  
count       30000.000000  
mean            0.339410  
std             0.465877  
min            -2.000000  
25%             0.040952  
50%             0.084932  
75%             0.586922  
max             2.000000  


## ✂️ Step 9: Train/Test Split

We use a stratified split to preserve the ~22% default rate in both the
training and test sets — critical for imbalanced classification, otherwise
a random split could accidentally skew the class balance in either set.

In [16]:
from sklearn.model_selection import train_test_split

X = df.drop("default payment next month", axis=1)
y = df["default payment next month"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"\nTrain default rate: {y_train.mean():.4f}")
print(f"Test default rate:  {y_test.mean():.4f}")

Train shape: (24000, 28), Test shape: (6000, 28)

Train default rate: 0.2212
Test default rate:  0.2212


## 🤖 Step 10: Baseline XGBoost Model

Before tuning anything for the imbalanced target, we train a plain XGBoost
classifier with default settings. This gives us a reference point — every
technique we add afterward (class weighting, threshold tuning) should be
measured against this baseline to prove it's actually helping.

In [17]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve
)

baseline_model = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)
y_proba_baseline = baseline_model.predict_proba(X_test)[:, 1]

print("=== Baseline XGBoost — Classification Report ===")
print(classification_report(y_test, y_pred_baseline, target_names=["No Default", "Default"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_baseline):.4f}")

=== Baseline XGBoost — Classification Report ===
              precision    recall  f1-score   support

  No Default       0.84      0.94      0.88      4673
     Default       0.61      0.36      0.46      1327

    accuracy                           0.81      6000
   macro avg       0.73      0.65      0.67      6000
weighted avg       0.79      0.81      0.79      6000

ROC-AUC: 0.7617
